<a href="https://colab.research.google.com/github/melaniedaniel7/CFPB-Text-Analysis-Using-NLP-Techniques/blob/main/Text_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython.display import display, Markdown

# Use Markdown # syntax for large headers
display(Markdown("# Text Cleaning"))

# Text Cleaning

In [ ]:
# Import libraries and download NLTK resources for text cleaning
import nltk
# These libraries were not mentioned in phase 1 and were an additional add-on for efficient text preprocessing
import re
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag # Import for POS tagging

# Punkt tokenizer for word tokenization
nltk.download('punkt')
# Stopwords list in multiple languages
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError
nltk.download('wordnet') # Added to resolve LookupError for WordNetLemmatizer
nltk.download('averaged_perceptron_tagger') # Added for POS tagging
nltk.download('averaged_perceptron_tagger_eng') # Added to resolve LookupError for averaged_perceptron_tagger_eng

# Create a set of English stopwords for efficient lookup
stop_words = set(stopwords.words('english'))

# Lemmatizer converts words to their base/root form
lemmatizer = WordNetLemmatizer()

# Function to convert NLTK POS tags to WordNet POS tags
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return nltk.corpus.wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return nltk.corpus.wordnet.VERB
    elif treebank_tag.startswith('N'):
        return nltk.corpus.wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return nltk.corpus.wordnet.ADV
    else:
        return nltk.corpus.wordnet.NOUN # Default to noun if not found

# Function to clean the text by converting text to lowercase and removing punctuation
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        # Remove patterns like 'xxxx' and 'xxxxyear'
        text = re.sub(r'x{2,}', '', text) # This will remove 'xxxx' and 'xxxxyear' will become 'year'
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\s+', ' ', text)
        return text
    else:
        return text

# Function to tokenize the text into individual words
def tokenize_text(text):
    if isinstance(text, str):
        tokens = word_tokenize(text)
        # The previous filter for set(word.lower()) == {'x'} is now less critical
        # as most 'x' sequences are handled by clean_text, but can keep it as a safeguard.
        tokens = [
            word for word in tokens
            if word and not (len(word) == 1 and word.lower() == 'x') # Exclude single 'x' if it appears
        ]
        return tokens
    else:
        return []

# Function to remove stopwords from the tokenized text
def remove_stopwords(tokens):
    if isinstance(tokens, list):
        return [word for word in tokens if word not in stop_words]
    else:
        return tokens

# Function to apply lemmatization to the tokens with POS tagging
def lemmatize_tokens(tokens):
    if isinstance(tokens, list):
        lemmatized_words = []
        for word, tag in pos_tag(tokens):
            wntag = get_wordnet_pos(tag)
            lemmatized_words.append(lemmatizer.lemmatize(word, wntag))
        return lemmatized_words
    else:
        return tokens

# Test text cleaning techniques on a single consumer complaint before applying text cleaning to the entire dataset
# Used the same complaint example as earlier
sample = df['Consumer complaint narrative'].iloc[0]
print("Original:")
print(sample)
# Clean example complaint
cleaned = clean_text(sample)
print("\nCleaned:")
print(cleaned)
# Tokenize example complaint
tokens = tokenize_text(cleaned)
print("\nTokens:")
print(tokens)
# Remove stopwords from example complaint
filtered = remove_stopwords(tokens)
print("\nNO Stpwords:")
print(filtered)
# Lemmatize example complaint
final = lemmatize_tokens(filtered)
print("\nLemmatized:")
print(final)

In [ ]:
# Apply the functions to the DataFrame
# Clean the text
df['Cleaned_Complaint'] = df['Consumer complaint narrative'].apply(clean_text)
# Tokenize the cleaned text
df['Tokenized_Complaint'] = df['Cleaned_Complaint'].apply(tokenize_text)
# Remove stopwords from the tokenized text
df['No_Stopwords_Complaint'] = df['Tokenized_Complaint'].apply(remove_stopwords)
# Apply lemmatization to the tokenized words
df['Lemmatized_Complaint'] = df['No_Stopwords_Complaint'].apply(lemmatize_tokens)

# Display the original and lemmatized complaint narratives
display(df[['Consumer complaint narrative', 'Lemmatized_Complaint']].head())